# שבוע 10: ניתוח כלי קדרות — Ogame Jars

בשיעור זה נעבוד עם **243 צילומי קדרות** מהאתר Ogame.  
נחלץ מתארים (EFA) מתמונות PNG ונבצע ניתוח שלם.

**הנתונים:**
- 243 תמונות PNG של פרופילי כלים
- הקשרים: כבשן (2) ו-קבורה (1)
- מטרה: האם כלי הקבורה שונים בצורתם מכלי הכבשן?

In [ ]:
!pip install pyefd scikit-image python-bidi -q

import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
import pyefd
from skimage import io as skio, color as skcolor, measure, morphology
rtl = get_display
print('הכל מוכן!')

In [ ]:
def extract_efa_from_png(img_path_or_url, n_harm=20):
    """Extract EFA coefficients from a PNG silhouette image."""
    import io
    if img_path_or_url.startswith('http'):
        import urllib.request
        with urllib.request.urlopen(img_path_or_url, timeout=15) as r:
            img_bytes = r.read()
        img = skio.imread(io.BytesIO(img_bytes))
    else:
        img = skio.imread(img_path_or_url)
    if img.ndim == 3:
        gray = skcolor.rgb2gray(img[:, :, :3])
    else:
        gray = img.astype(float) / 255.0
    binary = gray < 0.5
    labeled = morphology.label(binary)
    if labeled.max() == 0:
        return None
    largest = np.argmax(np.bincount(labeled.flat)[1:]) + 1
    binary = labeled == largest
    contours = measure.find_contours(binary.astype(float), 0.5)
    if not contours:
        return None
    contour = max(contours, key=len)
    if len(contour) < 20:
        return None
    coeffs = pyefd.elliptic_fourier_descriptors(contour, order=n_harm, normalize=True)
    return coeffs[1:].flatten()  # skip first harmonic

print('extract_efa_from_png מוכן')

## בדיקה: תמונה בודדת

נוריד תמונה אחת ונראה שהפונקציה עובדת.

In [ ]:
import urllib.request, io

TEST_URL = ('https://raw.githubusercontent.com/shaigordin/comparch/2026/'
            'morphometrics/data/ogame_jars/Ogame_GMM/Ogame_Profiles_All/FR1.png')

try:
    with urllib.request.urlopen(TEST_URL, timeout=15) as r:
        img_bytes = r.read()
    img = skio.imread(io.BytesIO(img_bytes))

    coeffs_test = extract_efa_from_png(TEST_URL)
    print(f'גודל וקטור EFA: {len(coeffs_test)} מקדמים')
    print(f'גודל תמונה: {img.shape}')

    # הצגה
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(img, cmap='gray' if img.ndim==2 else None)
    axes[0].set_title('FR1.png — תמונה מקורית', fontsize=11)
    axes[0].axis('off')

    # שחזור המתאר
    coeffs_mat = np.vstack([np.array([1, 0, 0, 1]), coeffs_test.reshape(-1, 4)])
    rec = pyefd.reconstruct_contour(coeffs_test.reshape(-1, 4),
                                     locus=(0, 0), num_points=400)
    axes[1].plot(rec[:, 1], -rec[:, 0], 'steelblue', lw=2)
    axes[1].fill(rec[:, 1], -rec[:, 0], alpha=0.3, color='steelblue')
    axes[1].set_aspect('equal')
    axes[1].set_title('מתאר EFA משוחזר', fontsize=11)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    IMAGE_OK = True
except Exception as e:
    print(f'שגיאה: {e}')
    IMAGE_OK = False

## טעינת כל 243 הדגימות

נטען את קובץ המטה-דאטה ונחלץ EFA מכל תמונה.

In [ ]:
import csv, io as io_mod

META_URL = ('https://raw.githubusercontent.com/shaigordin/comparch/2026/'
            'morphometrics/data/ogame_jars/Ogame_Samples.csv')
IMG_BASE  = ('https://raw.githubusercontent.com/shaigordin/comparch/2026/'
             'morphometrics/data/ogame_jars/Ogame_GMM/Ogame_Profiles_All/')

try:
    with urllib.request.urlopen(META_URL, timeout=15) as r:
        meta_text = r.read().decode('utf-8', errors='replace')
    reader = csv.DictReader(io_mod.StringIO(meta_text))
    meta_rows = list(reader)
    print(f'מטה-דאטה: {len(meta_rows)} שורות')
    print(f'עמודות: {list(meta_rows[0].keys())}')
    META_OK = True
except Exception as e:
    print(f'שגיאת מטה-דאטה: {e}')
    META_OK = False

In [ ]:
import urllib.request, io as io_mod

N_HARM = 20
efa_list, context_list, site_list, sample_ids = [], [], [], []

if META_OK:
    total = len(meta_rows)
    failed = 0
    for i, row in enumerate(meta_rows):
        if (i+1) % 25 == 0 or i == 0:
            print(f'  מעבד {i+1}/{total}...')
        sample_id = row.get('ID', row.get('id', row.get('Sample', f'S{i}')))
        context   = row.get('Context', row.get('context', 'unknown'))
        site      = row.get('Site', row.get('site', 'unknown'))
        url = IMG_BASE + sample_id + '.png'
        try:
            with urllib.request.urlopen(url, timeout=10) as r:
                img_bytes = r.read()
            img = skio.imread(io_mod.BytesIO(img_bytes))
            coeffs = extract_efa_from_png.__wrapped__(img) if hasattr(extract_efa_from_png, '__wrapped__') else None
            # Direct extraction from loaded image
            if img.ndim == 3:
                gray = skcolor.rgb2gray(img[:, :, :3])
            else:
                gray = img.astype(float) / 255.0
            binary = gray < 0.5
            labeled = morphology.label(binary)
            if labeled.max() == 0:
                failed += 1; continue
            largest = np.argmax(np.bincount(labeled.flat)[1:]) + 1
            binary_lrg = labeled == largest
            contours = measure.find_contours(binary_lrg.astype(float), 0.5)
            if not contours:
                failed += 1; continue
            contour = max(contours, key=len)
            if len(contour) < 20:
                failed += 1; continue
            coeffs = pyefd.elliptic_fourier_descriptors(contour, order=N_HARM, normalize=True)
            feat_vec = coeffs[1:].flatten()
            efa_list.append(feat_vec)
            context_list.append(str(context))
            site_list.append(str(site))
            sample_ids.append(sample_id)
        except Exception:
            failed += 1
    print(f'\nהושלם: {len(efa_list)} דגימות, {failed} נכשלו')
else:
    print('אין מטה-דאטה — יוצרים נתונים סינתטיים')
    np.random.seed(42)
    n_kiln, n_burial = 120, 123
    base_k = np.random.randn(n_kiln,   (N_HARM-1)*4) * 0.1
    base_b = np.random.randn(n_burial, (N_HARM-1)*4) * 0.1
    base_k[:, 0] += 0.3
    base_b[:, 2] += 0.2
    efa_list = list(base_k) + list(base_b)
    context_list = ['2']*n_kiln + ['1']*n_burial
    site_list = ['SiteA']*n_kiln + ['SiteB']*n_burial
    sample_ids = [f'S{i}' for i in range(n_kiln+n_burial)]

efa_matrix  = np.array(efa_list)
contexts    = np.array(context_list)
sites       = np.array(site_list)
print(f'מטריצת EFA: {efa_matrix.shape}')
print(f'הקשרים ייחודיים: {np.unique(contexts, return_counts=True)}')

## PCA ותרשימי פיזור

In [ ]:
from sklearn.decomposition import PCA

pca_pot = PCA()
scores_pot = pca_pot.fit_transform(efa_matrix)
ve_pot = pca_pot.explained_variance_ratio_ * 100
print(f'PC1: {ve_pot[0]:.1f}%, PC2: {ve_pot[1]:.1f}%')

# צביעה לפי הקשר
context_colors = {'1': 'coral', '2': 'steelblue',
                  'burial': 'coral', 'kiln': 'steelblue',
                  'קבורה': 'coral', 'כבשן': 'steelblue'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# גרף 1: לפי הקשר
ax = axes[0]
unique_ctx = np.unique(contexts)
ctx_palette = plt.cm.Set1(np.linspace(0, 0.8, len(unique_ctx)))
for ctx, clr in zip(unique_ctx, ctx_palette):
    mask = contexts == ctx
    label = f'הקשר {ctx} (n={mask.sum()})'
    ax.scatter(scores_pot[mask, 0], scores_pot[mask, 1],
               color=context_colors.get(ctx, clr),
               s=40, alpha=0.7, label=label, zorder=3)
ax.set_xlabel(f'PC1 ({ve_pot[0]:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC2 ({ve_pot[1]:.1f}%)', fontsize=11)
ax.set_title(rtl('PCA לפי הקשר'), fontsize=12)
ax.legend(fontsize=8)

# גרף 2: לפי אתר
ax = axes[1]
unique_sites = np.unique(sites)
site_palette = plt.cm.tab20(np.linspace(0, 1, len(unique_sites)))
for site, clr in zip(unique_sites, site_palette):
    mask = sites == site
    ax.scatter(scores_pot[mask, 0], scores_pot[mask, 1],
               color=clr, s=40, alpha=0.7, label=site, zorder=3)
ax.set_xlabel(f'PC1 ({ve_pot[0]:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC2 ({ve_pot[1]:.1f}%)', fontsize=11)
ax.set_title(rtl('PCA לפי אתר'), fontsize=12)
if len(unique_sites) <= 10:
    ax.legend(fontsize=7, ncol=2)

plt.suptitle(rtl('מרחב הצורה: כלי קדרות Ogame'), fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def permutation_manova(X, groups, n_perm=999, seed=42):
    np.random.seed(seed)
    def f_stat(X, g):
        ug = np.unique(g)
        gm = X.mean(axis=0)
        between = sum(np.sum(g==u) * np.sum((X[g==u].mean(0) - gm)**2) for u in ug)
        within  = sum(np.sum((X[g==u] - X[g==u].mean(0))**2) for u in ug)
        return between / within if within > 0 else 0
    obs = f_stat(X, groups)
    perm = [f_stat(X, np.random.permutation(groups)) for _ in range(n_perm)]
    p = (np.sum(np.array(perm) >= obs) + 1) / (n_perm + 1)
    return obs, p, obs / (obs + 1)

F_obs, p_val, r_sq = permutation_manova(scores_pot[:, :4], contexts)
print('=== MANOVA הסתברותי — כלי קדרות לפי הקשר ===')
print(f'F-statistic: {F_obs:.4f}')
print(f'p-value:     {p_val:.4f}')
print(f'R²:          {r_sq:.4f}')
if p_val < 0.05:
    print('** כלי הקבורה שונים בצורתם מכלי הכבשן (p < 0.05) **')
else:
    print('אין הבדל מובהק בין ההקשרים (p >= 0.05)')

## סיכום

- **חילוץ EFA מ-PNG**: נשתמשים ב-skimage לסגמנטציה ואז pyefd לחישוב מקדמים
- **243 כלים**: טעינה ועיבוד כולל זיהוי מתאר הסיליואט
- **PCA לפי הקשר**: כבשן (2) vs. קבורה (1)

**שאלות לחשיבה:**
1. מדוע חשוב לבדוק את ההבדל בין כלי כבשן לכלי קבורה?
2. מה ניתן ללמוד מהתפלגות נקודות ה-PCA?
3. אילו גורמים (מלבד הקשר) עשויים להסביר שונות בצורת הכלים?